In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

##Brands

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))

In [0]:
df_silver = df_silver.withColumn(
    "brand_code",
    F.regexp_replace(F.col("brand_code"), r"[^a-zA-Z0-9]", "")
)

In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(to_replace = anomalies, subset = ["category_code"])

df_silver.select("category_code").distinct().show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## category

In [0]:
df_bronz_cat = spark.table(f'{catalog_name}.bronze.brz_category')
df_bronz_cat.show(10)

In [0]:
df_silv_cat = df_bronz_cat.dropDuplicates(["category_code"])
df_silv_cat = df_silv_cat.withColumn("category_code", F.upper(F.col("category_code")))

In [0]:
df_silv_cat.write.format('delta') \
    .mode('overwrite') \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

##Products

In [0]:
df_bronze_pro = spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count, column_count = df_bronze_pro.count(), len(df_bronze_pro.columns)

print(f"row count is {row_count}")
print(f"column_count is  {column_count}")

In [0]:
# display(df_bronze_pro.limit(5))


In [0]:
# df_bronze_pro.select("weight_grams").show(5, truncate=False)

In [0]:
df_silver_pro = df_bronze_pro.withColumn("weigh_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
# df_silver_pro.select("weight_grams").show(5, truncate=False)

In [0]:
# df_silver_pro.select("length_cm").show(3)

In [0]:
df_silver_pro.printSchema()

In [0]:
df_silver_pro = df_silver_pro.withColumn(
    "length_cm",
    F.regexp_replace("length_cm", ",", ".").cast("float")
)

In [0]:
# df_silver_pro.select("length_cm").show(3)

In [0]:
# df_silver_pro.select("category_code", "brand_code").show(2)

In [0]:
df_silver_pro = df_silver_pro.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)

In [0]:
# df_silver_pro.select("category_code", "brand_code").show(2)

In [0]:
# df_silver_pro.select("material").distinct().show()

In [0]:
# Fix spelling mistakes
df_silver_pro = df_silver_pro.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)


In [0]:
df_silver_pro = df_silver_pro.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
df_silver_pro.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

##Customers

In [0]:
df_bronze_cus = spark.read.table(f'{catalog_name}.bronze.brz_customers')


In [0]:
# display(df_bronze_cus.limit(5))

In [0]:
# cnt_nul_id= df_bronze_cus.filter(F.col("customer_id").isNull()).count()
# cnt_nul_phone= df_bronze_cus.filter(F.col("phone").isNull()).count()
# cnt_nul_country= df_bronze_cus.filter(F.col("country_code").isNull()).count()
# cnt_nul_state= df_bronze_cus.filter(F.col("state").isNull()).count()
# cnt_nul_country= df_bronze_cus.filter(F.col("country").isNull()).count()
# print(f"Null customer_id: {cnt_nul_id}")
# print(f"Null phone: {cnt_nul_phone}")
# print(f"Null country_code: {cnt_nul_country}")
# print(f"Null state: {cnt_nul_state}")

In [0]:
df_silv_cust = df_bronze_cus.dropna(subset = ["customer_id"])
df_silv_cust = df_silv_cust.fillna({"phone":"NA"})


In [0]:
# df_silv_cust.filter(F.col("phone")=="NA").show()

In [0]:
# df_silv_cust.filter(F.col("customer_id").isNull()).count()

In [0]:
df_silv_cust.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## Calender Date

In [0]:
df_bronze_dat = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze_dat.count(), len(df_bronze_dat.columns)

# # Print the results
# print(f"Row count: {row_count}")
# print(f"Column count: {column_count}")

# df_bronze_dat.show(3)

In [0]:
dup = df_bronze_dat.groupBy("date").count().filter("count > 1")

In [0]:
print(dup.count())

In [0]:
df_silv_dat = df_bronze_dat.dropDuplicates(["date"])

In [0]:
# Capitalize first letter of each word in day_name
df_silv_dat = df_silv_dat.withColumn("day_name", F.initcap(F.col("day_name")))

In [0]:
  # Convert negative to positive
  df_silv_dat = df_silv_dat.withColumn("week_of_year", F.abs(F.col("week_of_year")))
  

In [0]:
df_silv_dat = df_silv_dat.withColumnRenamed("week_of_year", "week")

In [0]:
df_silv_dat = df_silv_dat.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silv_dat = df_silv_dat.withColumn("week", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week"), F.lit("-"), F.col("year"))))

# df_silv_dat.show(3)

In [0]:
df_silv_dat = df_silv_dat.withColumn("month_number", F.month(F.col("date")))
df_silv_dat = df_silv_dat.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
display(df_silv_dat)

In [0]:
df_silv_dat.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")